In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")


In [ ]:
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

label_counts = Counter(dataset["label"])
print(f"Class counts: {dict(label_counts)}")


In [ ]:
sent1_list = dataset["sentence1"]
sent2_list = dataset["sentence2"]
labels = dataset["label"]

batch_size = 32
predictions = []
confidences = []
probabilities = []

for start_idx in range(0, len(dataset), batch_size):
    batch_s1 = sent1_list[start_idx:start_idx + batch_size]
    batch_s2 = sent2_list[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch_s1,
        batch_s2,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    probabilities.extend(probs.cpu().tolist())

print(f"Completed inference for {len(predictions)} validation examples.")


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

pred_label_counts = Counter(predictions)
avg_confidence = sum(confidences) / len(confidences)

print("Evaluation metrics on full validation split:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print()
print("Class counts:")
print(f"True labels      : {dict(label_counts)}")
print(f"Predicted labels : {dict(pred_label_counts)}")
print(f"Average confidence: {avg_confidence:.4f}")


In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

mistakes = []
for i in range(len(dataset)):
    if predictions[i] != labels[i]:
        row = dataset[i]
        mistakes.append({
            "idx": i,
            "true_label": labels[i],
            "pred_label": predictions[i],
            "confidence": confidences[i],
            "prob_not_paraphrase": probabilities[i][0],
            "prob_paraphrase": probabilities[i][1],
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"]
        })

mistakes = sorted(mistakes, key=lambda x: x["confidence"], reverse=True)
print(f"Total mistakes on full validation split: {len(mistakes)}")

num_examples_to_show = min(10, len(mistakes))
for i in range(num_examples_to_show):
    m = mistakes[i]
    print(f"Mistake {i + 1}")
    print(f"idx: {m['idx']}")
    print(f"true label: {m['true_label']} ({label_map[m['true_label']]})")
    print(f"pred label: {m['pred_label']} ({label_map[m['pred_label']]})")
    print(f"confidence: {m['confidence']:.4f}")
    print(f"p(not_paraphrase)={m['prob_not_paraphrase']:.4f} | p(paraphrase)={m['prob_paraphrase']:.4f}")
    print(f"sentence1: {m['sentence1']}")
    print(f"sentence2: {m['sentence2']}")
    print("-" * 100)


In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset=full_validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"true_count_not_paraphrase={label_counts.get(0, 0)}")
print(f"true_count_paraphrase={label_counts.get(1, 0)}")
print(f"pred_count_not_paraphrase={pred_label_counts.get(0, 0)}")
print(f"pred_count_paraphrase={pred_label_counts.get(1, 0)}")
print(f"average_confidence={avg_confidence:.4f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"num_mistakes={len(mistakes)}")
